# 02 Data Assessing - Assessment Terpisah per Dataset

Notebook ini membaca output gathering dari `data/interim/raw_separate/`, lalu melakukan assessment kualitas data **untuk masing-masing dari 8 dataset**. Tidak ada proses menggabungkan transaksi dari semua dataset.

Output utama notebook ini:
- `data/interim/assessment_reports_by_dataset.json`
- `data/interim/assessment_summary_by_dataset.csv`
- `reports/02_data_quality_report_by_dataset.txt`
- visualisasi kualitas data per dataset.

## Setup Environment dan Load Raw Dataset Terpisah

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, HTML

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

try:
    from fingo_ds1.config import INTERIM_DATA_PATH, REPORTS_PATH
except ImportError:
    INTERIM_DATA_PATH = project_root / 'data' / 'interim'
    REPORTS_PATH = project_root / 'reports'

interim_data_path = Path(INTERIM_DATA_PATH)
reports_path = Path(REPORTS_PATH)
raw_separate_path = interim_data_path / 'raw_separate'
figures_path = reports_path / 'figures'
figures_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

manifest_path = raw_separate_path / 'raw_dataset_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError('Manifest raw dataset tidak ditemukan. Jalankan 01_data_gathering.ipynb terlebih dahulu.')

with open(manifest_path, 'r', encoding='utf-8') as file:
    dataset_catalog = json.load(file)

raw_datasets = {}
for meta in dataset_catalog:
    csv_path = project_root / meta['raw_output_path']
    frame = pd.read_csv(csv_path, low_memory=False)
    raw_datasets[meta['dataset_id']] = frame
    print(f"Loaded {meta['dataset_id']}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")

print(f'\nTotal dataset loaded: {len(raw_datasets)}')

Loaded ecommerce_2024_01_january: 431 rows x 24 columns
Loaded ecommerce_2024_06_june: 697 rows x 24 columns
Loaded ecommerce_2024_12_december: 1,214 rows x 23 columns
Loaded ecommerce_2025_02_february: 957 rows x 24 columns
Loaded ecommerce_2025_07_july: 766 rows x 23 columns
Loaded ecommerce_2025_11_november: 1,131 rows x 24 columns
Loaded daily_household_transactions: 2,461 rows x 14 columns
Loaded personal_finance: 1,500 rows x 11 columns

Total dataset loaded: 8


## Helper Assessment
Helper ini mendeteksi kolom penting, menghitung missing values, duplikasi, validitas tanggal, kualitas amount, dan outlier IQR untuk tiap dataset.

In [2]:
COLUMN_CANDIDATES = {
    'date': ['date', 'waktu_pesanan_dibuat', 'transaction_date', 'order_date'],
    'amount': ['amount', 'total_pembayaran', 'total_payment', 'payment_amount'],
    'category': ['category', 'product_categories', 'product_category', 'subcategory'],
    'description': ['transaction_description', 'description', 'note', 'order_id'],
    'transaction_type': ['type', 'income_expense', 'income/expense', 'status_pesanan'],
    'status': ['status_pesanan'],
    'payment_method': ['metode_pembayaran', 'mode', 'payment_method'],
    'city': ['kota_kabupaten', 'city'],
    'province': ['provinsi', 'province'],
    'quantity': ['total_qty', 'quantity'],
    'order_id': ['order_id'],
}


def normalize_name(value):
    text = str(value).strip().lower()
    text = re.sub(r'[/\\]+', '_', text)
    text = re.sub(r'[^0-9a-zA-Z]+', '_', text)
    return text.strip('_')


def find_column(frame, candidates):
    lookup = {normalize_name(column): column for column in frame.columns}
    normalized_candidates = [normalize_name(candidate) for candidate in candidates]

    for candidate in normalized_candidates:
        if candidate in lookup:
            return lookup[candidate]

    for candidate in normalized_candidates:
        for normalized_column, original_column in lookup.items():
            if candidate and candidate in normalized_column:
                return original_column

    return None


def detect_columns(frame):
    return {name: find_column(frame, candidates) for name, candidates in COLUMN_CANDIDATES.items()}


def parse_mixed_dates(series):
    values = series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    try:
        parsed = pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        parsed = pd.to_datetime(values, errors='coerce', dayfirst=True)

    numeric_values = pd.to_numeric(values, errors='coerce')
    excel_serial_mask = parsed.isna() & numeric_values.between(20000, 80000)
    if excel_serial_mask.any():
        parsed.loc[excel_serial_mask] = pd.to_datetime(
            numeric_values.loc[excel_serial_mask], unit='D', origin='1899-12-30', errors='coerce'
        )

    return parsed


def parse_amount(series):
    values = series.astype('string').str.strip()
    values = values.str.replace(r'\s+', '', regex=True)
    values = values.str.replace(r'(?<=\d),(?=\d{3}(\D|$))', '', regex=True)
    values = values.str.replace(',', '.', regex=False)
    values = values.str.replace(r'[^0-9.\-]', '', regex=True)
    values = values.replace({'': np.nan, 'nan': np.nan, 'None': np.nan, '<NA>': np.nan})
    return pd.to_numeric(values, errors='coerce')


def build_missing_summary(frame):
    if frame.empty:
        return pd.DataFrame(columns=['column', 'missing_count', 'missing_pct', 'present_count'])

    missing = frame.isna().sum()
    return (
        pd.DataFrame(
            {
                'column': frame.columns,
                'missing_count': missing.values.astype(int),
                'missing_pct': (missing / len(frame) * 100).round(2).values,
                'present_count': (len(frame) - missing.values).astype(int),
            }
        )
        .sort_values(['missing_pct', 'missing_count'], ascending=False)
        .reset_index(drop=True)
    )


def detect_outliers_iqr(series):
    values = series.dropna()
    if len(values) < 4:
        return pd.Series(False, index=series.index), np.nan, np.nan

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return (series < lower_bound) | (series > upper_bound), lower_bound, upper_bound


def to_python(value):
    if isinstance(value, dict):
        return {key: to_python(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_python(item) for item in value]
    if isinstance(value, tuple):
        return [to_python(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return None if pd.isna(value) else value.isoformat()
    if pd.isna(value):
        return None
    return value


def period_to_series(meta, index):
    period = meta.get('dataset_period')
    if not period:
        return pd.Series(pd.NaT, index=index)
    return pd.Series(pd.to_datetime(f'{period}-01', errors='coerce'), index=index)

## Jalankan Assessment untuk Tiap Dataset

In [3]:
def assess_dataset(dataset_id, frame, meta):
    detected = detect_columns(frame)
    missing_summary = build_missing_summary(frame)

    date_column = detected['date']
    if date_column:
        parsed_dates = parse_mixed_dates(frame[date_column])
        date_source = 'source_column'
    elif meta.get('dataset_period'):
        parsed_dates = period_to_series(meta, frame.index)
        date_source = 'dataset_period_from_filename'
    else:
        parsed_dates = pd.Series(pd.NaT, index=frame.index)
        date_source = 'not_available'

    amount_column = detected['amount']
    amount_values = parse_amount(frame[amount_column]) if amount_column else pd.Series(np.nan, index=frame.index)
    outlier_mask, outlier_lower, outlier_upper = detect_outliers_iqr(amount_values)

    category_column = detected['category']
    category_values = frame[category_column].astype('string').str.strip() if category_column else pd.Series(pd.NA, index=frame.index)
    top_categories = category_values.value_counts(dropna=True).head(10).to_dict()

    key_columns = [column for column in [date_column, amount_column, category_column] if column]
    subset_duplicates = int(frame.duplicated(subset=key_columns, keep=False).sum()) if len(key_columns) >= 2 else 0

    valid_dates = parsed_dates.dropna()
    valid_amounts = amount_values.dropna()
    today = pd.Timestamp.today().normalize()

    critical_missing = {
        'date': {
            'source': date_source,
            'count': int(parsed_dates.isna().sum()),
            'percentage': round(float(parsed_dates.isna().mean() * 100), 2) if len(frame) else 0.0,
        },
        'amount': {
            'source': amount_column,
            'count': int(amount_values.isna().sum()),
            'percentage': round(float(amount_values.isna().mean() * 100), 2) if len(frame) else 0.0,
        },
        'category': {
            'source': category_column,
            'count': int(category_values.isna().sum()),
            'percentage': round(float(category_values.isna().mean() * 100), 2) if len(frame) else 0.0,
        },
    }

    issue_summary = []
    if critical_missing['date']['percentage'] > 10:
        issue_summary.append('Tanggal banyak kosong/tidak valid')
    if critical_missing['amount']['percentage'] > 5:
        issue_summary.append('Amount banyak kosong/tidak valid')
    if int((amount_values < 0).sum()) > 0:
        issue_summary.append('Ada amount negatif')
    if subset_duplicates > 0:
        issue_summary.append('Ada potensi duplikasi berdasarkan key date-amount-category')
    if int(outlier_mask.sum()) > 0:
        issue_summary.append('Ada outlier amount berdasarkan IQR')
    if not issue_summary:
        issue_summary.append('Tidak ada issue besar yang terdeteksi otomatis')

    return {
        'dataset_info': {
            'dataset_id': dataset_id,
            'dataset_name': meta['dataset_name'],
            'domain': meta['domain'],
            'dataset_period': meta.get('dataset_period'),
            'source_path': meta['source_path'],
            'records': int(len(frame)),
            'columns': int(frame.shape[1]),
            'memory_mb': round(float(frame.memory_usage(deep=True).sum() / (1024 ** 2)), 4),
        },
        'schema': {
            'detected_columns': detected,
            'columns': list(frame.columns),
            'dtypes': {column: str(dtype) for column, dtype in frame.dtypes.items()},
        },
        'missing_values': {
            'total_missing_cells': int(frame.isna().sum().sum()),
            'columns_with_missing': int((frame.isna().sum() > 0).sum()),
            'top_missing_columns': missing_summary.head(12).to_dict(orient='records'),
            'critical_missing': critical_missing,
        },
        'date_quality': {
            'date_column': date_column,
            'date_source': date_source,
            'unparseable_dates': int(parsed_dates.isna().sum()),
            'future_dates': int((parsed_dates > today).sum()),
            'min_date': valid_dates.min() if not valid_dates.empty else None,
            'max_date': valid_dates.max() if not valid_dates.empty else None,
        },
        'amount_quality': {
            'amount_column': amount_column,
            'missing_amounts': int(amount_values.isna().sum()),
            'negative_amounts': int((amount_values < 0).sum()),
            'zero_amounts': int((amount_values == 0).sum()),
            'min': valid_amounts.min() if not valid_amounts.empty else None,
            'median': valid_amounts.median() if not valid_amounts.empty else None,
            'mean': valid_amounts.mean() if not valid_amounts.empty else None,
            'max': valid_amounts.max() if not valid_amounts.empty else None,
        },
        'duplicates': {
            'exact_duplicates': int(frame.duplicated().sum()),
            'subset_columns': key_columns,
            'subset_duplicates': subset_duplicates,
            'subset_duplicate_pct': round(float(subset_duplicates / len(frame) * 100), 2) if len(frame) else 0.0,
        },
        'outliers': {
            'method': 'IQR amount',
            'lower_bound': outlier_lower,
            'upper_bound': outlier_upper,
            'count': int(outlier_mask.sum()),
            'percentage': round(float(outlier_mask.sum() / len(frame) * 100), 2) if len(frame) else 0.0,
        },
        'category_quality': {
            'category_column': category_column,
            'unique_categories': int(category_values.nunique(dropna=True)),
            'top_categories': top_categories,
        },
        'issue_summary': issue_summary,
    }


assessment_reports = {}
summary_rows = []

for meta in dataset_catalog:
    dataset_id = meta['dataset_id']
    report = assess_dataset(dataset_id, raw_datasets[dataset_id], meta)
    assessment_reports[dataset_id] = report

    summary_rows.append(
        {
            'dataset_id': dataset_id,
            'dataset_name': meta['dataset_name'],
            'records': report['dataset_info']['records'],
            'columns': report['dataset_info']['columns'],
            'missing_cells': report['missing_values']['total_missing_cells'],
            'columns_with_missing': report['missing_values']['columns_with_missing'],
            'date_missing_pct': report['missing_values']['critical_missing']['date']['percentage'],
            'amount_missing_pct': report['missing_values']['critical_missing']['amount']['percentage'],
            'category_missing_pct': report['missing_values']['critical_missing']['category']['percentage'],
            'exact_duplicates': report['duplicates']['exact_duplicates'],
            'subset_duplicates': report['duplicates']['subset_duplicates'],
            'outlier_pct': report['outliers']['percentage'],
            'issue_count': len(report['issue_summary']),
        }
    )

assessment_summary_df = pd.DataFrame(summary_rows).sort_values('dataset_id').reset_index(drop=True)
display(assessment_summary_df)

,dataset_id,dataset_name,records,columns,missing_cells,columns_with_missing,date_missing_pct,amount_missing_pct,category_missing_pct,exact_duplicates,subset_duplicates,outlier_pct,issue_count
0,daily_household_transactions,Daily Household Transactions,2461,14,3617,3,0.0,0.0,0.0,9,99,12.80,2
1,ecommerce_2024_01_january,E-Commerce Sales - January 2024,431,24,367,1,0.0,0.0,0.0,0,2,8.35,2
2,ecommerce_2024_06_june,E-Commerce Sales - June 2024,697,24,608,1,0.0,0.0,0.0,0,0,10.33,1
3,ecommerce_2024_12_december,E-Commerce Sales - December 2024,1214,23,1047,1,0.0,0.0,0.0,0,559,9.64,2
4,ecommerce_2025_02_february,E-Commerce Sales - February 2025,957,24,835,1,0.0,0.0,0.0,0,0,12.02,1
5,ecommerce_2025_07_july,E-Commerce Sales - July 2025,766,23,676,1,0.0,0.0,0.0,0,374,12.01,2
6,ecommerce_2025_11_november,E-Commerce Sales - November 2025,1131,24,1030,1,0.0,0.0,0.0,0,2,11.05,2
7,personal_finance,Personal Finance Dataset,1500,11,1500,1,0.0,0.0,0.0,0,0,5.87,1


## Detail Assessment Tiap Dataset
Bagian ini menampilkan kolom yang terdeteksi, missing values terbesar, kategori teratas, dan issue prioritas untuk masing-masing dataset.

In [4]:
for dataset_id, report in assessment_reports.items():
    print('\n' + '=' * 100)
    print(f"{dataset_id} - {report['dataset_info']['dataset_name']}")
    print('=' * 100)
    print('Detected columns:')
    display(pd.DataFrame(report['schema']['detected_columns'].items(), columns=['standard_field', 'detected_column']))

    print('Top missing columns:')
    display(pd.DataFrame(report['missing_values']['top_missing_columns']).head(8))

    print('Top categories:')
    display(pd.Series(report['category_quality']['top_categories'], name='count').to_frame())

    print('Issue summary:')
    for issue in report['issue_summary']:
        print(f'- {issue}')


ecommerce_2024_01_january - E-Commerce Sales - January 2024
Detected columns:


,standard_field,detected_column
0,date,Waktu Pesanan Dibuat
1,amount,Total Pembayaran
2,category,product_categories
3,description,order_id
4,transaction_type,Status Pesanan
5,status,Status Pesanan
6,payment_method,Metode Pembayaran
7,city,Kota/Kabupaten
8,province,Provinsi
9,quantity,total_qty


Top missing columns:


,column,missing_count,missing_pct,present_count
0,Alasan Pembatalan,367,85.15,64
1,order_id,0,0.00,431
2,total_qty,0,0.00,431
3,total_weight_gr,0,0.00,431
4,total_returned_qty,0,0.00,431
5,Total Diskon,0,0.00,431
6,product_categories,0,0.00,431
7,num_product_categories,0,0.00,431


Top categories:


...

Top missing columns:


,column,missing_count,missing_pct,present_count
0,_dataset_period,1500,100.0,0
1,Date,0,0.0,1500
2,Transaction Description,0,0.0,1500
3,Category,0,0.0,1500
4,Amount,0,0.0,1500
5,Type,0,0.0,1500
6,_dataset_id,0,0.0,1500
7,_dataset_name,0,0.0,1500


Top categories:


,count
Rent,165
Travel,160
Utilities,157
Health & Fitness,152
Shopping,150
Food & Drink,149
Salary,146
Entertainment,143
Investment,142
Other,136


Issue summary:
- Ada outlier amount berdasarkan IQR


## Save Assessment Report Terpisah

In [5]:
def build_text_report(reports):
    lines = [
        '=' * 100,
        'DATA QUALITY ASSESSMENT REPORT - 8 DATASET TERPISAH',
        '=' * 100,
        '',
    ]

    for dataset_id, report in reports.items():
        info = report['dataset_info']
        lines.extend(
            [
                f"DATASET: {dataset_id}",
                '-' * 100,
                f"Name            : {info['dataset_name']}",
                f"Rows x Columns  : {info['records']:,} x {info['columns']:,}",
                f"Date Source     : {report['date_quality']['date_source']}",
                f"Date Range      : {to_python(report['date_quality']['min_date'])} to {to_python(report['date_quality']['max_date'])}",
                f"Missing Cells   : {report['missing_values']['total_missing_cells']:,}",
                f"Amount Missing  : {report['missing_values']['critical_missing']['amount']['percentage']}%",
                f"Exact Duplicate : {report['duplicates']['exact_duplicates']:,}",
                f"Key Duplicate   : {report['duplicates']['subset_duplicates']:,}",
                f"Outlier Amount  : {report['outliers']['count']:,} ({report['outliers']['percentage']}%)",
                'Issues:',
            ]
        )
        lines.extend(f"- {issue}" for issue in report['issue_summary'])
        lines.append('')

    lines.append('=' * 100)
    return '\n'.join(lines)

assessment_json_path = interim_data_path / 'assessment_reports_by_dataset.json'
assessment_summary_path = interim_data_path / 'assessment_summary_by_dataset.csv'
text_report_path = reports_path / '02_data_quality_report_by_dataset.txt'

with open(assessment_json_path, 'w', encoding='utf-8') as file:
    json.dump(to_python(assessment_reports), file, indent=2)
assessment_summary_df.to_csv(assessment_summary_path, index=False)
text_report_path.write_text(build_text_report(assessment_reports), encoding='utf-8')

print('Assessment outputs saved:')
print(f'- {assessment_json_path.relative_to(project_root)}')
print(f'- {assessment_summary_path.relative_to(project_root)}')
print(f'- {text_report_path.relative_to(project_root)}')

Assessment outputs saved:
- data/interim/assessment_reports_by_dataset.json
- data/interim/assessment_summary_by_dataset.csv
- reports/02_data_quality_report_by_dataset.txt


## Visualisasi Kualitas Data
Dashboard ini membandingkan metrik kualitas dari 8 dataset berdasarkan summary assessment, bukan gabungan transaksi.

In [6]:
from html import escape


def build_bar_chart_html(frame, label_col, value_col, title, color, value_suffix='', value_format=',.2f'):
    chart = frame[[label_col, value_col]].copy()
    chart[value_col] = pd.to_numeric(chart[value_col], errors='coerce').fillna(0)
    max_value = chart[value_col].max() or 1
    rows = []

    for _, row in chart.iterrows():
        label = escape(str(row[label_col]))
        value = float(row[value_col])
        width = max(2, value / max_value * 100)
        value_text = format(value, value_format) + value_suffix
        rows.append(
            f'''<div class="bar-row">
                <div class="bar-label">{label}</div>
                <div class="bar-track"><div class="bar-fill" style="width:{width:.2f}%; background:{color};"></div></div>
                <div class="bar-value">{value_text}</div>
            </div>'''
        )

    return f'''<section class="chart-card">
        <h3>{escape(title)}</h3>
        {''.join(rows)}
    </section>'''


plot_df = assessment_summary_df.sort_values('records', ascending=True).copy()
plot_df['short_name'] = plot_df['dataset_id'].str.replace('ecommerce_', 'ecom_', regex=False)
plot_df['missing_cell_pct'] = (plot_df['missing_cells'] / (plot_df['records'] * plot_df['columns']) * 100).replace([np.inf, -np.inf], np.nan).fillna(0)

html = f'''
<style>
.dashboard-grid {{ display: grid; grid-template-columns: repeat(2, minmax(300px, 1fr)); gap: 16px; font-family: Arial, sans-serif; }}
.chart-card {{ border: 1px solid #d8dee4; border-radius: 8px; padding: 14px; background: #ffffff; }}
.chart-card h3 {{ margin: 0 0 12px 0; font-size: 16px; color: #1f2937; }}
.bar-row {{ display: grid; grid-template-columns: minmax(140px, 1.5fr) 2fr minmax(80px, .8fr); gap: 8px; align-items: center; margin: 8px 0; }}
.bar-label {{ font-size: 12px; color: #374151; overflow-wrap: anywhere; }}
.bar-track {{ height: 14px; background: #eef2f7; border-radius: 999px; overflow: hidden; }}
.bar-fill {{ height: 100%; border-radius: 999px; }}
.bar-value {{ font-size: 12px; color: #111827; text-align: right; font-variant-numeric: tabular-nums; }}
@media (max-width: 900px) {{ .dashboard-grid {{ grid-template-columns: 1fr; }} }}
</style>
<div class="dashboard-grid">
{build_bar_chart_html(plot_df, 'short_name', 'records', 'Jumlah Record per Dataset', '#2f6f73', value_format=',.0f')}
{build_bar_chart_html(plot_df, 'short_name', 'missing_cell_pct', 'Persentase Missing Cell', '#b54d4d', '%')}
{build_bar_chart_html(plot_df, 'short_name', 'subset_duplicates', 'Potensi Duplikasi Key', '#6f5f90', value_format=',.0f')}
{build_bar_chart_html(plot_df, 'short_name', 'outlier_pct', 'Outlier Amount Berdasarkan IQR', '#bf7b45', '%')}
</div>
'''

dashboard_path = figures_path / '02_data_assessing_quality_dashboard.html'
dashboard_path.write_text(html, encoding='utf-8')
display(HTML(html))
print(f'Dashboard saved: {dashboard_path.relative_to(project_root)}')


Dashboard saved: reports/figures/02_data_assessing_quality_dashboard.html


## Output Assessment
Tahap assessment selesai ketika report per dataset sudah tersimpan. Notebook cleaning akan memakai `assessment_reports_by_dataset.json` untuk menentukan kolom tanggal, amount, kategori, dan field penting lain secara terpisah.